In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, confusion_matrix, precision_score, recall_score 
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, 
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC


In [2]:
df = pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/train.csv")



In [3]:
#train-test split
x = df.drop(['Irrigation_Need'], axis =1)
y = df['Irrigation_Need']
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size = 0.2, random_state = 42
)

In [11]:
from sklearn.preprocessing import KBinsDiscretizer
numeric_columns = df.select_dtypes(include= ['int64', 'float64']).columns

categorical_columns = df.select_dtypes(include = ['object']).columns
categorical_columns = categorical_columns.drop('Irrigation_Need', errors='ignore')

binning = KBinsDiscretizer(n_bins = 10, encode = 'ordinal', strategy = 'quantile')


In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline

preprocessor = ColumnTransformer(
    transformers = [
        ('binning', binning, numeric_columns),
        ('onehot',OneHotEncoder(sparse_output = False),categorical_columns),
        
    ],
    remainder = 'passthrough'
)

 
pipeline = Pipeline([('encoding', preprocessor),('model',RandomForestClassifier(n_estimators =100, random_state=42))])

pipeline.fit(x_train, y_train)
y_pred = pipeline.predict(x_test)

In [13]:
print("BalancedAccuracy:", balanced_accuracy_score(y_test, y_pred) * 100)
print("Confusion Matrix:\n",
confusion_matrix(y_test, y_pred))
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

print("Precision:", precision)
print("Recall:", recall)

BalancedAccuracy: 88.94137321796805
Confusion Matrix:
 [[ 3095    22  1132]
 [    0 73272   465]
 [  136  2450 45428]]
Precision: 0.9665401902136792
Recall: 0.9666269841269841
